# CSCU9M6 Assignment


In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

## Deliverable 1: Data Loading and Preprocessing

In [7]:
ROOT = Path.cwd()
TRAIN_CSV = ROOT / "sign_mnist_train" / "sign_mnist_train.csv"
TEST_CSV = ROOT / "sign_mnist_test" / "sign_mnist_test.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"dataset labels: {sorted(train_df['label'].unique())}")

dataset labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]


### Fix the labels so that they are 0-23 rather than 0-24 and missing 9

In [8]:
def remap_labels(label):
    if label >= 10:
        return label - 1
    else:
        return label

train_df['label'] = train_df['label'].apply(remap_labels)
test_df['label'] = test_df['label'].apply(remap_labels)

TRAIN_CSV_UPDATED = ROOT / "sign_mnist_train" / "sign_mnist_train_updated.csv"
TEST_CSV_UPDATED = ROOT / "sign_mnist_test" / "sign_mnist_test_updated.csv"

train_df.to_csv(TRAIN_CSV_UPDATED, index=False)
test_df.to_csv(TEST_CSV_UPDATED, index=False)

In [11]:
class SignLanguageDataset(Dataset):
    def __init__(self, dataframe):
        self.labels = dataframe['label'].astype(int).to_numpy()
        self.pixels = dataframe.drop(columns=['label']).values.astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.pixels[idx], dtype=torch.float32).view(1, 28, 28)
        x = x / 255.0 # normalize pixels to between 0 and 1
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

In [12]:
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=1,
    stratify=train_df['label']
    )

train_dataset = SignLanguageDataset(train_split_df)
val_dataset = SignLanguageDataset(val_split_df)
test_dataset = SignLanguageDataset(test_df)

print('Train size:', len(train_dataset))
print('Val size:', len(val_dataset))
print('Test size:', len(test_dataset))

Train size: 24709
Val size: 2746
Test size: 7172


## Deliverable 2: CNN Architectures
- **Baseline CNN**: Simple CNN using 2 conv layers, 2 pooling layers, and 2 fully connected layers.
- **CNN with Residual Blocks**: Uses skip connections to allow for a deeper network without a vanishing gradient.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=24):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()

        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu2 = nn.ReLU()

    def forward(self, x):
        res = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = x + res
        x = self.relu2(x)
        return x


class ResidualCNN(nn.Module):
    def __init__(self, num_classes=24, blocks=2):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.res_blocks1 = nn.ModuleList([ResBlock(32) for _ in range(blocks)])

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.res_blocks2 = nn.ModuleList([ResBlock(64) for _ in range(blocks)])

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()
        self.res_blocks3 = nn.ModuleList([ResBlock(128) for _ in range(blocks)])

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        for block in self.res_blocks1:
            x = block(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        for block in self.res_blocks2:
            x = block(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        for block in self.res_blocks3:
            x = block(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

## Deliverable 3: Training, Testing, and Evaluation